In [1]:
import os

In [2]:
%pwd

'd:\\STUDY\\CDS\\End-to-End-ML-project-with-MLFlow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\STUDY\\CDS\\End-to-End-ML-project-with-MLFlow'

In [7]:
# 4. Entity Updated
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [ ]:
# 5. Update the configuration manager in src config
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [ ]:
# from mlProject.components import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH

# import importlib
# import mlProject.utils.common as common
# importlib.reload(common)

# read_yaml = common.read_yaml
# create_directories = common.create_directories

In [9]:
# 5. Update the configuration manager in src config
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        paramas_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(paramas_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
        )

        return data_ingestion_config

In [10]:
# 6. Update the components
import os
import urllib.request as request # For downloading data from URL
import zipfile # Used to ZIP the data
from mlProject import logger
from mlProject.utils.common import get_size # TO get file size

In [ ]:
# 6. Update the components
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file,
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(self.config.local_data_file)}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [ ]:
# 7. Update the pipeline and Data Ingetion
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-05-21 09:52:34,817: INFO: common: yaml file: config\config.yaml Loaded successfully]
[2026-05-21 09:52:34,821: INFO: common: yaml file: params.yaml Loaded successfully]
[2026-05-21 09:52:34,825: INFO: common: yaml file: schema.yaml Loaded successfully]
[2026-05-21 09:52:34,828: INFO: common: created directory at: artifacts]
[2026-05-21 09:52:34,833: INFO: common: created directory at: artifacts/data_ingestion]
[2026-05-21 09:52:36,229: INFO: 2753784264: artifacts/data_ingestion/data.zip downloaded! with following info: 
Connection: close
Content-Length: 24777
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "0006f9650554e67dc021c528af5c386ac92a3dbca6f826707425341a50c3f4d1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 5334:D6B40:1D74A7:469DC2:6A0E857C
Accept-Ranges: bytes
Date: Thu, 2